In [3]:
'''
Function: ner_zero_shot
Description:
 - Performs zero-shot Named Entity Recognition (NER) using the GPT-4o model.
 - Accepts a system instruction, input text, and temperature value as parameters.
 - Sends the instruction and formatted text to the OpenAI chat API.
 - Extracts and cleans the model’s response.
 - Attempts to parse the response as JSON.
 - Returns the parsed output if valid, otherwise returns None.
'''

def ner_zero_shot(instruction, text, temperature):
    client = OpenAI()
    user_query = "TEXT: {text}"
    
    response = client.chat.completions.create(
      model = 'gpt-4o',
      temperature = temperature,
      messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_query.format(text=text)}
      ]
    )

    content = response.choices[0].message.content
    cleaned_content = content.strip("```python\n").strip("```")
    
    try:
        output = json.loads(cleaned_content)
        return output
    except json.JSONDecodeError as e:
        return None

In [ ]:
## THIS FUNCTION IS ONLY NEEDED FOR CONTINUOUS ANNOTATION

def entity_formatter(terms_in_paragraph):
    columns = ['paragraph', 'start_index', 'end_index', 'label', 'llm_term', 'sliced_term']
    df = pd.DataFrame(terms_in_paragraph, columns=columns)
    df_sorted = df.sort_values(by="start_index")
    df_sorted.reset_index(drop=True, inplace=True)

    df_sorted["equal"] = df_sorted["llm_term"] == df_sorted["sliced_term"]
    df_sorted["overlap"] = False

    for i in range(1, len(df_sorted)):
        if df_sorted.loc[i, "start_index"] >= df_sorted.loc[i-1, "start_index"] and \
            df_sorted.loc[i, "start_index"] <= df_sorted.loc[i-1, "end_index"]:
            df_sorted.loc[i, "overlap"] = True

    df_sorted = df_sorted[df_sorted["equal"] != False]
    df_sorted = df_sorted[df_sorted["overlap"] != True]
    # df_sorted = df_sorted[df_sorted['equal']]  # Keep only where equal is True
    # df_sorted = df_sorted[~df_sorted['overlap']]  # Remove rows with overlap

    # df_sorted

    output_list = []

    for index, row in df_sorted.iterrows():
        output_list.append([row["start_index"], row["end_index"], row["label"]])

    final_output = {"entities": output_list}

    return final_output

In [40]:
'''
Function: evaluate_distinct_entities
Description:
 - Evaluates Named Entity Recognition (NER) performance at both paragraph and document levels.
 - Accepts predicted terms, gold standard terms, and corresponding paragraphs for comparison.
 - Calculates precision, recall, and F1-score for each label in each paragraph.
 - Aggregates entity sets across paragraphs to compute overall metrics per label.
 - Saves detailed outputs including:
   - A .txt file logging paragraph-level predictions and gold terms.
   - An Excel file for paragraph-level performance.
   - An Excel file for overall (document-level) performance.
'''

from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from evaluation_variables_t1r1 import labels, paragraphs, gold_standard_terms, predicted_terms
from chatgpt_prompt_COPIED import instructions

# NEED PYTHON 3.9 OR MORE FOR TYPE DESCRIPTION
# def evaluate_distinct_entities(
#     labels: list[str],
#     paragraphs: list[str],
#     gold_standard_entities: list[dict[str, list[str]]],
#     predicted_entities: list[dict[str, list[str]]],
#     download_location: str
# ) -> None:
    
def evaluate_distinct_entities(
    labels: list,
    paragraphs: list,
    gold_standard_entities: list,
    predicted_entities: list,
    download_location: str
) -> None:
    
    # paragraph-level performance calculation 
    # variable declaration
    all_gs_entities = {
        'chemical': set(),
        'material': set(),
        'structure': set(),
        'property': set(),
        'application': set(),
        'process': set(),
        'equipment': set(),
        'measurement': set(),
        'abbreviation': set()
    }
    all_pred_entities = {
        'chemical': set(),
        'material': set(),
        'structure': set(),
        'property': set(),
        'application': set(),
        'process': set(),
        'equipment': set(),
        'measurement': set(),
        'abbreviation': set()
    }
    df_score_para = pd.DataFrame(columns=['metrics', 'values'])
    eval_metrics_para = []
    eval_values_para = []
    
    # retrieving paragraph, gold standard terms, and predicted terms from zipped list
    for para, gs_ent, pred_ent in zip(paragraphs, gold_standard_entities, predicted_entities):
        paragraph_index = paragraphs.index(para)
        
        # writing paragraph number, paragraph, gold standard terms, and predicted terms in a text file
        with open(f'{download_location}\\paragraphs-with-all-terms.txt', 'a', encoding='utf-8') as file:
            file.write(f'PARAGRAPH NUMBER: {paragraph_index}\n')
            file.write(f'PARAGRAPH: {para}\n')
            file.write(f'GOLD STANDARD TERMS: {gs_ent}\n')
            file.write(f'PREDICTED TERMS: {pred_ent}\n')
            file.write('=======================================================\n')
        
        print(f'Updated paragraphs-with-all-terms.txt in {download_location}')
        
        # retrieving labels from list
        for label in labels:
            
            # creating entity set with unique entities
            unique_gs_ent = set(gs_ent[label])
            unique_pred_ent = set(pred_ent[label])
            
            # storing entities (label-wise) for document-level calculation
            all_gs_entities[label].update(unique_gs_ent)
            all_pred_entities[label].update(unique_pred_ent)
            
            # calculate confusion matrix
            tp = unique_gs_ent & unique_pred_ent
            fp = unique_pred_ent - unique_gs_ent
            fn = unique_gs_ent - unique_pred_ent
            
            # calculate precision, recall and f1-score for each paragraph
            precision = len(tp) / (len(tp) + len(fp)) if unique_pred_ent else 0
            recall = len(tp) / (len(tp) + len(fn)) if unique_gs_ent else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            # storing performance for each paragraph in list
            eval_metrics_para.extend([f'{paragraph_index} {label.upper()} Precision', 
                                      f'{paragraph_index} {label.upper()} Recall', 
                                      f'{paragraph_index} {label.upper()} F1'])

            eval_values_para.extend([f'{precision:.2f}',
                                     f'{recall:.2f}', 
                                     f'{f1:.2f}'])
    
    # creating dataframe from list
    df_score_para['metrics'] = eval_metrics_para
    df_score_para['values'] = eval_values_para

    # downloading paragraph-level performace in a spreadsheet
    df_score_para.to_excel(f'{download_location}\\score-para-DIS-ENTS.xlsx', index=False)
    print(f'Downloaded score-para-DIS-ENTS.xlsx in {download_location}')
    
    # document-level performance calculation  
    # variable declaration
    df_score_doc = pd.DataFrame(columns=['metrics', 'values'])
    eval_metrics_doc = []
    eval_values_doc = []
    
    # retrieving label from list
    for label in labels:

        # calculate confusion matrix
        tp = all_gs_entities[label] & all_pred_entities[label]
        fp = all_pred_entities[label] - all_gs_entities[label]
        fn = all_gs_entities[label] - all_pred_entities[label]
        
        # calculate precision, recall and f1-score for entire document
        precision = len(tp) / (len(tp) + len(fp)) if all_pred_entities[label] else 0
        recall = len(tp) / (len(tp) + len(fn)) if all_gs_entities[label] else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f'{label}:\t{f1:.2f}(f) | {precision:.2f}(p) | {recall:.2f}(r)')
        
        # storing performance for the entire document in list
        eval_metrics_doc.extend([f'{label.upper()} (Overall) Precision', 
                                f'{label.upper()} (Overall) Recall', 
                                f'{label.upper()} (Overall) F1'])
        
        eval_values_doc.extend([f'{precision:.2f}',
                               f'{recall:.2f}', 
                               f'{f1:.2f}'])

    # creating dataframe from list
    df_score_doc['metrics'] = eval_metrics_doc
    df_score_doc['values'] = eval_values_doc

    # downloading document-level performace in a spreadsheet
    df_score_doc.to_excel(f'{download_location}\\score-doc-DIS-ENTS.xlsx', index=False)
    print(f'Downloaded score-doc-DIS-ENTS.xlsx in {download_location}')
    
#     # document-level analysis
#     # overlap between chemical and material entities
#     print('Overlap between chemical and material entities:')
#     overlap_in_gold_standard = len(all_gs_entities['chemical'] & all_gs_entities['material'])
#     overlap_in_predicted = len(all_pred_entities['chemical'] & all_pred_entities['material'])
#     print(f'In gold standard data: {overlap_in_gold_standard}')
#     print(f'In predicted data: {overlap_in_predicted}')
    
#     # overlap between chemical and structure entities
#     print('Overlap between chemical and structure entities:')
#     overlap_in_gold_standard = len(all_gs_entities['chemical'] & all_gs_entities['structure'])
#     overlap_in_predicted = len(all_pred_entities['chemical'] & all_pred_entities['structure'])
#     print(f'In gold standard data: {overlap_in_gold_standard}')
#     print(f'In predicted data: {overlap_in_predicted}')
    
    
#     # overlap between material and structure entities
#     print('Overlap between chemical and structure entities:')
#     overlap_in_gold_standard = len(all_gs_entities['material'] & all_gs_entities['structure'])
#     overlap_in_predicted = len(all_pred_entities['material'] & all_pred_entities['structure'])
#     print(f'In gold standard data: {overlap_in_gold_standard}')
#     print(f'In predicted data: {overlap_in_predicted}')
    
#     print(all_gs_entities['chemical'] & all_gs_entities['material'], all_pred_entities['chemical'] & all_pred_entities['material'])
#     print(all_gs_entities['chemical'] & all_gs_entities['structure'], all_pred_entities['chemical'] & all_pred_entities['structure'])
#     print(all_gs_entities['material'] & all_gs_entities['structure'], all_pred_entities['material'] & all_pred_entities['structure'])
    

In [4]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from collections import Counter

def evaluate_all_entities(
    labels: list,
    paragraphs: list,
    gs_entities: list,
    pred_entities: list
) -> None:
    
    # paragraph-level performance calculation 
    # variable declaration
    all_gs_entities = {
        'chemical': Counter(),
        'material': Counter(),
        'structure': Counter(),
        'property': Counter(),
        'application': Counter(),
        'process': Counter(),
        'equipment': Counter(),
        'measurement': Counter(),
        'abbreviation': Counter()
    }
    all_pred_entities = {
        'chemical': Counter(),
        'material': Counter(),
        'structure': Counter(),
        'property': Counter(),
        'application': Counter(),
        'process': Counter(),
        'equipment': Counter(),
        'measurement': Counter(),
        'abbreviation': Counter()
    }
    df_score_para = pd.DataFrame(columns=['metrics', 'values'])
    eval_metrics_para = []
    eval_values_para = []
    
    # retrieving paragraph, gold standard terms, and predicted terms from zipped list
    for para, gs_ent, pred_ent in zip(paragraphs, gs_entities, pred_entities):
        paragraph_index = paragraphs.index(para)
        
        # writing paragraph number, paragraph, gold standard terms, and predicted terms in a text file
#         with open(f'{download_location}\\paragraphs-with-all-terms.txt', 'a', encoding='utf-8') as file:
#             file.write(f'PARAGRAPH NUMBER: {paragraph_index}\n')
#             file.write(f'PARAGRAPH: {para}\n')
#             file.write(f'GOLD STANDARD TERMS: {gs_ent}\n')
#             file.write(f'PREDICTED TERMS: {pred_ent}\n')
#             file.write('=======================================================\n')
        
#         print(f'Updated paragraphs-with-all-terms.txt in {download_location}')
        
        # retrieving labels from list
        for label in labels:
            
            # count occurrence of each term
            gs_counter = Counter(gs_ent[label])
            pred_counter = Counter(pred_ent[label])
            
            # storing entities (label-wise) for document-level calculation
            all_gs_entities[label].update(gs_counter)  # CHECK: IF SAME TERM COMES FROM 2ND PARAGRAPH
            all_pred_entities[label].update(pred_counter)
            
            # calculate confusion matrix
            tp_counter = gs_counter & pred_counter  # Intersection of counts
            tp_sum = sum(tp_counter.values())

            fp_counter = pred_counter - gs_counter  # Predicted but not in gold
            fp_sum = sum(fp_counter.values())

            fn_counter = gs_counter - pred_counter  # Gold but not in predicted
            fn_sum = sum(fn_counter.values())

            # calculate precision, recall and f1-score for each paragraph
            precision = tp_sum / (tp_sum + fp_sum) if pred_counter else 0
            recall = tp_sum / (tp_sum + fn_sum) if gs_counter else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            # storing performance for each paragraph in list
            eval_metrics_para.extend([f'{paragraph_index} {label.upper()} Precision', 
                                      f'{paragraph_index} {label.upper()} Recall', 
                                      f'{paragraph_index} {label.upper()} F1'])

            eval_values_para.extend([f'{precision:.2f}',
                                     f'{recall:.2f}', 
                                     f'{f1:.2f}'])
    
    # creating dataframe from list
    df_score_para['metrics'] = eval_metrics_para
    df_score_para['values'] = eval_values_para

    # downloading paragraph-level performace in a spreadsheet
    df_score_para.to_excel(f'{download_location}\\score-para-ALL-ENTS.xlsx', index=False)
    print(f'Downloaded score-para-ALL-ENTS.xlsx in {download_location}')

    # document-level performance calculation  
    # variable declaration
    df_score_doc = pd.DataFrame(columns=['metrics', 'values'])
    eval_metrics_doc = []
    eval_values_doc = []
    
    # ADDED FOR OVERLAP
    tp_sum_all_labels, fp_sum_all_labels, fn_sum_all_labels = 0, 0, 0
    
    # retrieving label from list
    for label in labels:

        # calculate confusion matrix
        tp_counter = all_gs_entities[label] & all_pred_entities[label]
        tp_sum = sum(tp_counter.values())

        fp_counter = all_pred_entities[label] - all_gs_entities[label]
        fp_sum = sum(fp_counter.values())

        fn_counter = all_gs_entities[label] - all_pred_entities[label]
        fn_sum = sum(fn_counter.values())
        
        # calculate precision, recall and f1-score for entire document
        precision = tp_sum / (tp_sum + fp_sum) if all_pred_entities[label] else 0
        recall = tp_sum / (tp_sum + fn_sum) if all_gs_entities[label] else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        print(f'{label}:\t{f1:.2f}(f) | {precision:.2f}(p) | {recall:.2f}(r)')
        
        # storing performance for the entire document in list
        eval_metrics_doc.extend([f'{label.upper()} (Overall) Precision', 
                                f'{label.upper()} (Overall) Recall', 
                                f'{label.upper()} (Overall) F1'])
        
        eval_values_doc.extend([f'{precision:.2f}',
                               f'{recall:.2f}', 
                               f'{f1:.2f}'])
    
    # creating dataframe from list
    df_score_doc['metrics'] = eval_metrics_doc
    df_score_doc['values'] = eval_values_doc

    # downloading document-level performace in a spreadsheet
    df_score_doc.to_excel(f'{download_location}\\score-doc-ALL-ENTS.xlsx', index=False)
    print(f'Downloaded score-doc-ALL-ENTS.xlsx in {download_location}')
       

In [7]:
from evaluation_variables_t1r1 import labels, paragraphs, gold_standard_terms, predicted_terms
# evaluate_all_entities(labels, paragraphs, gold_standard_terms, predicted_terms)

for gs_ent in gold_standard_terms:
    for label in labels:
        tt = Counter(gs_ent[label])
        print(tt)
        print('------------------')

Counter()
------------------
Counter({'cotton fibers': 1, 'bast fibers': 1, 'wood': 1, 'composite materials': 1, 'cotton': 1})
------------------
Counter({'cellulose microfibrils': 1, 'cellulosic man-made fibers': 1, 'noncrystalline chain': 1, 'man-made cellulose fibers': 1, 'technical fibers': 1, 'bast fibers': 1})
------------------
Counter({'helix angle': 2, 'orientation': 2, 'lower orientation': 1, 'module of elasticity': 1, 'elongation at breakage': 1, 'microfibril orientation': 1, 'fiber strength': 1, 'mechanical properties': 1, 'helix angles': 1, 'elongation': 1, 'modulus of elasticity': 1})
------------------
Counter()
------------------
Counter({'targeted influence': 1})
------------------
Counter({'cellulose phenylurethanes': 1, 'cellulose-(3-chlorophenyl)-urethanes': 1, 'Cellulose urethanes': 1, 'azo-dye': 1, 'acrylates of hydroxypropyl cellulose': 1, 'lyotropic liquid-crystalline cellulose': 1})
------------------
Counter({'water': 2, 'gels': 1})
------------------
Counter(

## *Main:* (for performance evaluation)

#### <span style="color: blue;">Description:</span>
- calls 
    - ***ner_zero_shot()*** 
    - ***performance_metrics()***
- generates files through *performance_metrics()*
    - *dl-all-terms.txt*
    - *dl-performance-paragraph.xlsx*
    - *dl-performance-overall.xlsx*
- generates files *evaluation_variables.py*

#### <span style="color: red;">Check:</span>
- *file_path* for evaluation data
- *labels* in case of merging or changing labels
- argument value (to set temperature) in *annotator_without_examples()* calling
- *download_location* for saving files

In [ ]:
'''
Script Description:
 - Reads NER evaluation data from a text file.
 - Separates paragraphs and gold standard terms.
 - Parses the gold terms from JSON strings into dictionaries.
 - Performs zero-shot NER prediction using GPT-4o for each paragraph.
 - Ensures that predictions include all expected labels.
 - Saves evaluation variables to a file for reproducibility.
 - Evaluates model performance using a custom evaluation function.
'''


# READ INPUT FILE
file_path = '..\\OpenAI-Fine-tuning\\data\\test-data.txt'
with open(file_path, 'r', encoding='utf-8') as file:
    file_list = file.read().splitlines()
    
# STORE PARAGRAPHS AND ANNOTATIONS IN DIFFERENT LISTS
paragraphs = []
gold_standard_terms_str = []

for item in file_list:
    if file_list.index(item) == 0 or file_list.index(item) % 2 == 0:
        paragraphs.append(item)
    else:
        gold_standard_terms_str.append(item)
        
# CONVERT ANNOTATIONS TO NESTED OBJECTS FROM STRING
gold_standard_terms = []

for item in gold_standard_terms_str:
    try:
        json_obj = json.loads(item)             # Convert to dictionary
        gold_standard_terms.append(json_obj)    # Add to list of dictionaries
    except json.JSONDecodeError as e:
        print(f'Error decoding JSON for item: {item}\nError: {e}')

# SEND PARAGRAPHS AND INSTRUCTIONS TO LLM 
labels = [
    'chemical',
    'material',
    'structure',
    'property',
    'application',
    'process',
    'equipment',
    'measurement',
    'abbreviation',
  ]

predicted_terms = []

# predict terms from each paragraph for every instruction

for paragraph in paragraphs:
    data = dict()

    for instruction in instructions:
        response = ner_zero_shot(instruction, paragraph, 0.3)    # calling annotator()
        if response:
            label = list(response.keys())
            terms = list(response.values())
            data[label[0]] = terms[0]
#         else:
#             print("ERROR::", response)
    
    # check missing labels in predicted data
    if len(data) < len(labels):
        
        print('Label missing in predicted data.')
        
        temp_data = dict()
        
        for label in labels:
            if label not in data:
                data[label] = []
                print(f'Label -- {label} -- added to predicted data.')
        
        ## organize the dictionary keys according to label's order
        for label in labels:
            temp_data[label] = data[label]
            
        data = temp_data
    
    predicted_terms.append(data)

# save labels, paragraphs, gold_standard_terms, predicted_terms variables

download_location = '..\\OpenAI-Fine-tuning\\test-cases\\test-case-3\\run-2'

with open(f'{download_location}\\evaluation_variables.py', 'w', encoding='utf-8') as file:
    file.write('labels = ' + repr(labels) + '\n')
    file.write('paragraphs = ' + repr(paragraphs) + '\n')
    file.write('gold_standard_terms = ' + repr(gold_standard_terms) + '\n')
    file.write('predicted_terms = ' + repr(predicted_terms) + '\n')
    
print(f'Downloaded evaluation_variables.py in {download_location}')
    
evaluate_performance(labels, paragraphs, gold_standard_terms, predicted_terms, download_location)
